In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
import lightgbm as lgb
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

# ==================== HELPER FUNCTIONS ====================

def remove_emojis(text):
    if pd.isna(text):
        return ""
    text = str(text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002500-\U00002BEF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002600-\U000026FF"
        "\U00002700-\U000027BF"
        "\U0001F018-\U0001F270"
        "\U0001F300-\U0001F5FF"
        "\U0000FE00-\U0000FE0F"
        "\U0001F680-\U0001F6FF"
        "]+",
        flags=re.UNICODE
    )
    cleaned = emoji_pattern.sub('', text)
    cleaned = ' '.join(cleaned.split())
    return cleaned.strip()

def extract_fields(catalog_content):
    if pd.isna(catalog_content):
        return pd.Series([None, None, None, ""])
    
    catalog_content = str(catalog_content)
    
    item_name_match = re.search(
        r'Item Name: (.+?)(?=Bullet Point|Product Description|Value:|$)',
        catalog_content, re.IGNORECASE | re.DOTALL
    )
    item_name = item_name_match.group(1).strip() if item_name_match else None
    
    value_match = re.search(r'Value: ([\d.]+)', catalog_content, re.IGNORECASE)
    try:
        value = float(value_match.group(1).strip()) if value_match else None
    except (ValueError, AttributeError):
        value = None
    
    unit_match = re.search(r'Unit: (.+?)$', catalog_content, re.IGNORECASE)
    unit = unit_match.group(1).strip() if unit_match else None
    
    catalog_content_cleaned = re.sub(
        r'Item Name: .*?(?=Bullet Point|Product Description|Value:|$)|Value: .*?(?=Unit:|$)|Unit: .*?$',
        '', catalog_content, flags=re.IGNORECASE | re.DOTALL
    ).strip()
    
    return pd.Series([item_name, value, unit, catalog_content_cleaned])

def standardize_units(row):
    unit = row['unit']
    value = row['value']
    
    if pd.isna(unit) or pd.isna(value):
        return pd.Series({'standardized_value': value, 'standardized_unit': unit})
    
    try:
        value_float = float(value)
    except:
        return pd.Series({'standardized_value': value, 'standardized_unit': unit})
    
    unit_lower = str(unit).lower().strip()
    
    if unit_lower in ['ounce', 'ounces', 'oz']:
        return pd.Series({'standardized_value': round(value_float * 28.3495, 2), 'standardized_unit': 'grams'})
    elif unit_lower in ['gram', 'grams', 'g']:
        return pd.Series({'standardized_value': value_float, 'standardized_unit': 'grams'})
    elif unit_lower in ['kilogram', 'kilograms', 'kg', 'kgs']:
        return pd.Series({'standardized_value': round(value_float * 1000, 2), 'standardized_unit': 'grams'})
    elif unit_lower in ['fl oz', 'fluid ounce', 'fluid ounces', 'floz', 'fl. oz', 'fluid ounce(s)']:
        return pd.Series({'standardized_value': round(value_float * 29.5735, 2), 'standardized_unit': 'ml'})
    elif unit_lower in ['ml', 'millilitre', 'milliliter', 'millilitres', 'milliliters']:
        return pd.Series({'standardized_value': value_float, 'standardized_unit': 'ml'})
    elif unit_lower in ['liter', 'liters', 'litre', 'litres', 'l']:
        return pd.Series({'standardized_value': round(value_float * 1000, 2), 'standardized_unit': 'ml'})
    elif unit_lower in ['pound', 'lb', 'lbs']:
        return pd.Series({'standardized_value': round(value_float * 453.592, 2), 'standardized_unit': 'grams'})
    elif unit_lower in ['count', 'ct']:
        return pd.Series({'standardized_value': value_float, 'standardized_unit': 'Count'})
    else:
        return pd.Series({'standardized_value': value_float, 'standardized_unit': unit})

def remove_punctuation(text):
    if isinstance(text, str):
        return re.sub(r'[^\w\s]', ' ', text).strip()
    return ""

def count_bullet_points(catalog_content):
    if isinstance(catalog_content, str):
        matches = re.findall(r'Bullet Point\s*\d*:', catalog_content, re.IGNORECASE)
        return len(matches)
    return 0

def remove_bullet_point_text_only(catalog_content):
    if isinstance(catalog_content, str):
        return re.sub(r'Bullet Point\s*\d*:', '', catalog_content, flags=re.IGNORECASE).strip()
    return ""

def extract_premium_features(item_name, desc):
    combined = ""
    if isinstance(item_name, str):
        combined += item_name.lower() + " "
    if isinstance(desc, str):
        combined += desc.lower()
    
    premium_strict = r'pr[ea]mi?u?m|gourmet|artisan|luxury|deluxe|supreme|elite|finest|signature|reserve|hand\s*crafted|handmade'
    premium_count_terms = r'pr[ea]mi?u?m|gourmet|artisan|luxury|deluxe|supreme|hand\s*(?:made|crafted)|extra\s*virgin|imported|select|choice|prime|signature'
    
    is_premium_strict = int(bool(re.search(premium_strict, combined)))
    premium_word_count = len(re.findall(premium_count_terms, combined))
    is_super_premium = int(premium_word_count >= 2)
    
    if is_premium_strict == 1:
        premium_tier = 2
    elif premium_word_count > 0:
        premium_tier = 1
    else:
        premium_tier = 0
    
    return pd.Series({
        'is_premium_strict': is_premium_strict,
        'premium_word_count': premium_word_count,
        'premium_tier': premium_tier,
        'is_super_premium': is_super_premium
    })

def extract_freshness_features(item_name, desc):
    combined = ""
    if isinstance(item_name, str):
        combined += item_name.lower() + " "
    if isinstance(desc, str):
        combined += desc.lower()
    
    fresh_pattern = r'fresh|farm\s*fresh|garden\s*fresh|refrigerated|chilled'
    natural_pattern = r'natural|all\s*natural|pure|raw|unprocessed|whole'
    
    return pd.Series({
        'is_fresh': int(bool(re.search(fresh_pattern, combined))),
        'is_natural': int(bool(re.search(natural_pattern, combined)))
    })

def extract_dietary_features(item_name, desc):
    combined = ""
    if isinstance(item_name, str):
        combined += item_name.lower() + " "
    if isinstance(desc, str):
        combined += desc.lower()
    
    return pd.Series({
        'low_fat': int(bool(re.search(r'low\s*fat|fat\s*free|reduced\s*fat|lite|light', combined))),
        'organic': int(bool(re.search(r'organic|usda\s*certified\s*organic', combined))),
        'vegan': int(bool(re.search(r'vegan|plant\s*based', combined))),
        'caffeine_free': int(bool(re.search(r'caffeine\s*free|decaf', combined)))
    })

def extract_additional_features(item_name, desc):
    combined = ""
    if isinstance(item_name, str):
        combined += item_name.lower() + " "
    if isinstance(desc, str):
        combined += desc.lower()
    
    return pd.Series({
        'is_bulk': int(bool(re.search(r'bulk|family\s*size|party\s*size|economy\s*size', combined))),
        'is_frozen': int(bool(re.search(r'frozen|freeze', combined))),
        'is_spicy': int(bool(re.search(r'spicy|hot|chili|pepper', combined))),
        'is_mild': int(bool(re.search(r'mild|gentle', combined))),
        'is_canned': int(bool(re.search(r'canned|tinned|can', combined))),
        'is_new': int(bool(re.search(r'new|introducing|just\s*launched', combined))),
        'is_extra_flavor': int(bool(re.search(r'extra|double|triple|loaded', combined))),
        'is_creamy': int(bool(re.search(r'creamy|smooth|silky|rich', combined))),
        'is_multipack': int(bool(re.search(r'pack\s*of|multipack|\d+\s*pack', combined))),
        'is_roasted': int(bool(re.search(r'roasted|roast', combined))),
        'is_imported': int(bool(re.search(r'imported|import|from\s*italy|from\s*france', combined)))
    })

def extract_additional_features_2(item_name, desc):
    combined = ""
    if isinstance(item_name, str):
        combined += item_name.lower() + " "
    if isinstance(desc, str):
        combined += desc.lower()
    
    return pd.Series({
        'is_raw': int(bool(re.search(r'raw|uncooked', combined))),
        'is_award_winning': int(bool(re.search(r'award|winning|gold\s*medal', combined))),
        'is_low_calorie': int(bool(re.search(r'low\s*calorie|lite|diet', combined))),
        'is_processed': int(bool(re.search(r'processed|refined', combined))),
        'is_liquid': int(bool(re.search(r'liquid|drink|beverage|juice', combined))),
        'is_sour': int(bool(re.search(r'sour|tart|tangy|citrus', combined))),
        'is_sweet': int(bool(re.search(r'sweet|honey|sugar|caramel|chocolate', combined))),
        'is_aged': int(bool(re.search(r'aged|\d+\s*year\s*old|matured', combined))),
        'is_soft': int(bool(re.search(r'soft|tender|moist', combined))),
        'is_bitter': int(bool(re.search(r'bitter|dark\s*chocolate|coffee', combined))),
        'is_restaurant_style': int(bool(re.search(r'restaurant\s*style|chef|gourmet\s*style', combined)))
    })

def extract_text_features(item_name, desc):
    combined_text = ""
    if isinstance(item_name, str):
        combined_text += item_name.lower() + " "
    if isinstance(desc, str):
        combined_text += desc.lower()
    
    if not combined_text.strip():
        return pd.Series({
            'text_length': 0, 'word_count': 0, 'unique_words': 0,
            'avg_word_length': 0, 'char_count': 0, 'digit_count': 0,
            'upper_count': 0, 'special_char_count': 0,
            'sentence_count': 0, 'lexical_diversity': 0
        })
    
    original_text = (item_name if isinstance(item_name, str) else "") + " " + (desc if isinstance(desc, str) else "")
    words = combined_text.split()
    unique_words = len(set(words))
    word_count = len(words)
    
    return pd.Series({
        'text_length': len(combined_text),
        'word_count': word_count,
        'unique_words': unique_words,
        'avg_word_length': np.mean([len(w) for w in words]) if words else 0,
        'char_count': len(combined_text.replace(" ", "")),
        'digit_count': sum(c.isdigit() for c in combined_text),
        'upper_count': sum(c.isupper() for c in original_text),
        'special_char_count': len(re.findall(r'[^a-zA-Z0-9\s]', combined_text)),
        'sentence_count': len(re.split(r'[.!?]+', combined_text)),
        'lexical_diversity': unique_words / word_count if word_count > 0 else 0
    })

custom_stop_words = set(ENGLISH_STOP_WORDS) | {
    'product', 'description', 'item', 'pack', 'oz', 'ounce', 'count',
    'and', 'or', 'the', 'a', 'an', 'of', 'for', 'with', 'to', 'in', 'on'
}

def remove_stopwords(text):
    if not isinstance(text, str):
        return ""
    words = text.lower().split()
    filtered_words = [word for word in words if word not in custom_stop_words and len(word) > 2]
    return ' '.join(filtered_words)

def calculate_smape(actual, predicted):
    """Calculate SMAPE metric"""
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted)))

# ==================== MAIN PREPROCESSING FUNCTION ====================

def preprocess_data(df, is_train=True, scaler=None, tfidf_name=None, tfidf_desc=None, 
                    svd_name=None, svd_desc=None, train_median_value=None):
    """Main preprocessing pipeline"""
    
    print(f"Starting preprocessing... Shape: {df.shape}")
    initial_count = len(df)
    
    df['catalog_content'] = df['catalog_content'].apply(remove_emojis)
    df[['item_name', 'value', 'unit', 'catalog_content_cleaned']] = df['catalog_content'].apply(extract_fields)
    df[['standardized_value', 'standardized_unit']] = df.apply(standardize_units, axis=1)
    
    if is_train:
        print(f"Before filtering: {len(df)} rows")
        df = df[~df['standardized_unit'].isin(['none', 'each', 'packs', 'pack', 'bottle'])]
        df = df.dropna(subset=['standardized_unit'])
        df = df[df['standardized_unit'].isin(['grams', 'ml', 'Count'])]
        df = df.dropna(subset=['item_name'])
        print(f"After filtering: {len(df)} rows")
        train_median_value = df['standardized_value'].median()
    else:
        df['standardized_unit'] = df['standardized_unit'].fillna('grams')
        df['standardized_value'] = df['standardized_value'].fillna(train_median_value)
        df['item_name'] = df['item_name'].fillna('')
    
    df = pd.get_dummies(df, columns=['standardized_unit'], prefix='unit', dtype=int)
    
    for col in ['unit_Count', 'unit_grams', 'unit_ml']:
        if col not in df.columns:
            df[col] = 0
    
    df = df.drop(['value', 'unit', 'catalog_content'], axis=1, errors='ignore')
    
    df['has_catalog_content'] = ((~df['catalog_content_cleaned'].isna()) & 
                                  (df['catalog_content_cleaned'].str.strip() != '')).astype(int)
    
    if is_train:
        scaler = MinMaxScaler()
        df['value_scaled'] = scaler.fit_transform(df[['standardized_value']])
    else:
        df['value_scaled'] = scaler.transform(df[['standardized_value']])
    
    df = df.drop(columns=['standardized_value'], errors='ignore')
    
    df['bullet_point_count'] = df['catalog_content_cleaned'].apply(count_bullet_points)
    df['catalog_content'] = df['catalog_content_cleaned'].apply(remove_bullet_point_text_only)
    df['desc'] = df['catalog_content']
    df = df.drop(columns=['catalog_content_cleaned', 'catalog_content', 'image_link'], errors='ignore')
    
    df['item_name'] = df['item_name'].apply(remove_punctuation)
    df['desc'] = df['desc'].apply(remove_punctuation)
    
    premium_df = df.apply(lambda row: extract_premium_features(row['item_name'], row['desc']), axis=1)
    df = pd.concat([df, premium_df], axis=1)
    
    fresh_df = df.apply(lambda row: extract_freshness_features(row['item_name'], row['desc']), axis=1)
    df = pd.concat([df, fresh_df], axis=1)
    
    dietary_df = df.apply(lambda row: extract_dietary_features(row['item_name'], row['desc']), axis=1)
    df = pd.concat([df, dietary_df], axis=1)
    
    add_df = df.apply(lambda row: extract_additional_features(row['item_name'], row['desc']), axis=1)
    df = pd.concat([df, add_df], axis=1)
    
    add_df_2 = df.apply(lambda row: extract_additional_features_2(row['item_name'], row['desc']), axis=1)
    df = pd.concat([df, add_df_2], axis=1)
    
    text_df = df.apply(lambda row: extract_text_features(row['item_name'], row['desc']), axis=1)
    df = pd.concat([df, text_df], axis=1)
    
    df['item_name'] = df['item_name'].fillna('').apply(remove_stopwords)
    df['desc'] = df['desc'].fillna('').apply(remove_stopwords)
    
    if is_train:
        tfidf_name = TfidfVectorizer(max_features=200, min_df=2, max_df=0.95, 
                                     ngram_range=(1, 2), sublinear_tf=True)
        tfidf_name_matrix = tfidf_name.fit_transform(df['item_name'].fillna(''))
        
        svd_name = TruncatedSVD(n_components=30, random_state=42)
        name_embeddings = svd_name.fit_transform(tfidf_name_matrix)
        
        tfidf_desc = TfidfVectorizer(max_features=300, min_df=2, max_df=0.95, 
                                     ngram_range=(1, 2), sublinear_tf=True)
        tfidf_desc_matrix = tfidf_desc.fit_transform(df['desc'].fillna(''))
        
        svd_desc = TruncatedSVD(n_components=40, random_state=42)
        desc_embeddings = svd_desc.fit_transform(tfidf_desc_matrix)
    else:
        tfidf_name_matrix = tfidf_name.transform(df['item_name'].fillna(''))
        name_embeddings = svd_name.transform(tfidf_name_matrix)
        
        tfidf_desc_matrix = tfidf_desc.transform(df['desc'].fillna(''))
        desc_embeddings = svd_desc.transform(tfidf_desc_matrix)
    
    name_cols = [f'name_emb_{i}' for i in range(30)]
    name_df = pd.DataFrame(name_embeddings, columns=name_cols, index=df.index)
    
    desc_cols = [f'desc_emb_{i}' for i in range(40)]
    desc_df = pd.DataFrame(desc_embeddings, columns=desc_cols, index=df.index)
    
    df = pd.concat([df, name_df, desc_df], axis=1)
    df = df.drop(columns=['item_name', 'desc'], errors='ignore')
    
    final_count = len(df)
    print(f"Preprocessing complete. Final shape: {df.shape}")
    
    if not is_train:
        if final_count != initial_count:
            print(f"WARNING: Lost {initial_count - final_count} test rows!")
        else:
            print(f"✓ All {initial_count} test rows preserved")
    
    if is_train:
        return df, scaler, tfidf_name, tfidf_desc, svd_name, svd_desc, train_median_value
    else:
        return df

# ==================== MAIN EXECUTION ====================

print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

print("\nPreprocessing training data...")
train_processed, scaler, tfidf_name, tfidf_desc, svd_name, svd_desc, train_median_value = preprocess_data(
    train_df.copy(), is_train=True
)

print("\nPreprocessing test data...")
test_processed = preprocess_data(
    test_df.copy(), is_train=False, scaler=scaler, 
    tfidf_name=tfidf_name, tfidf_desc=tfidf_desc,
    svd_name=svd_name, svd_desc=svd_desc,
    train_median_value=train_median_value
)

feature_cols = [col for col in train_processed.columns 
                if col not in ['sample_id', 'price']]

X_train = train_processed[feature_cols].fillna(0)
y_train = train_processed['price']
X_test = test_processed[feature_cols].fillna(0)

print(f"\nFeature count: {len(feature_cols)}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# ==================== LIGHTGBM & XGBOOST ENSEMBLE ====================

print("\n" + "="*60)
print("Training LightGBM & XGBoost Ensemble")
print("="*60)

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

# Optimized LightGBM parameters
lgb_params = {
    'objective': 'regression',
    'metric': 'mape',
    'boosting_type': 'gbdt',
    'learning_rate': 0.02,
    'num_leaves': 31,
    'max_depth': 8,
    'min_child_samples': 20,
    'subsample': 0.8,
    'subsample_freq': 1,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'min_split_gain': 0.01,
    'verbose': -1,
    'random_state': 42,
    'n_jobs': -1
}

# Optimized XGBoost parameters
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mape',
    'learning_rate': 0.02,
    'max_depth': 7,
    'min_child_weight': 3,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'tree_method': 'hist',
    'n_jobs': -1
}

# Store predictions
lgb_test_preds = []
lgb_train_preds = np.zeros(len(X_train))

xgb_test_preds = []
xgb_train_preds = np.zeros(len(X_train))

# Cross-validation for LightGBM
print("\n--- Training LightGBM ---")
lgb_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=2000,
        valid_sets=[val_data],
        callbacks=[lgb.early_stopping(stopping_rounds=100), lgb.log_evaluation(100)]
    )
    
    val_pred = model.predict(X_val, num_iteration=model.best_iteration)
    lgb_train_preds[val_idx] = val_pred
    
    smape = calculate_smape(y_val, val_pred)
    lgb_fold_scores.append(smape)
    print(f"Fold {fold+1} SMAPE: {smape:.4f}")

print(f"LightGBM Average SMAPE: {np.mean(lgb_fold_scores):.4f} (+/- {np.std(lgb_fold_scores):.4f})")

# Train LightGBM on full data
print("\nTraining LightGBM on full training data...")
full_train_data = lgb.Dataset(X_train, label=y_train)
lgb_model = lgb.train(lgb_params, full_train_data, num_boost_round=2000)
lgb_test_pred = lgb_model.predict(X_test)

# Cross-validation for XGBoost
print("\n--- Training XGBoost ---")
xgb_fold_scores = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, 'eval')],
        early_stopping_rounds=100,
        verbose_eval=100
    )
    
    val_pred = model.predict(dval)
    xgb_train_preds[val_idx] = val_pred
    
    smape = calculate_smape(y_val, val_pred)
    xgb_fold_scores.append(smape)
    print(f"Fold {fold+1} SMAPE: {smape:.4f}")

print(f"XGBoost Average SMAPE: {np.mean(xgb_fold_scores):.4f} (+/- {np.std(xgb_fold_scores):.4f})")

# Train XGBoost on full data
print("\nTraining XGBoost on full training data...")
dtrain_full = xgb.DMatrix(X_train, label=y_train)
xgb_model = xgb.train(xgb_params, dtrain_full, num_boost_round=2000)
dtest = xgb.DMatrix(X_test)
xgb_test_pred = xgb_model.predict(dtest)

# Ensemble predictions (weighted average)
print("\n" + "="*60)
print("Creating Ensemble Predictions")
print("="*60)

# Optimal weights based on individual performance
lgb_weight = 0.55
xgb_weight = 0.45

final_train_pred = lgb_weight * lgb_train_preds + xgb_weight * xgb_train_preds
final_test_pred = lgb_weight * lgb_test_pred + xgb_weight * xgb_test_pred

train_smape = calculate_smape(y_train, final_train_pred)

print(f"\nIndividual Model Performance:")
print(f"  LightGBM CV SMAPE: {np.mean(lgb_fold_scores):.4f}")
print(f"  XGBoost CV SMAPE: {np.mean(xgb_fold_scores):.4f}")
print(f"\nEnsemble Performance:")
print(f"  Training SMAPE: {train_smape:.4f}")
print(f"  Weights: LightGBM={lgb_weight}, XGBoost={xgb_weight}")

# Create submission
submission = pd.DataFrame({
    'sample_id': test_processed['sample_id'],
    'price': final_test_pred
})

# Ensure prices are positive
submission['price'] = submission['price'].clip(lower=0.01)

print(f"\nSubmission shape: {submission.shape}")
print(f"Expected test samples: {len(test_df)}")
print(f"Actual predictions: {len(submission)}")

if len(submission) == len(test_df):
    print("✓ All test samples have predictions!")
else:
    print(f"✗ WARNING: Missing {len(test_df) - len(submission)} predictions!")

print(f"\nPrediction stats:")
print(f"  Min: ${submission['price'].min():.2f}")
print(f"  Max: ${submission['price'].max():.2f}")
print(f"  Mean: ${submission['price'].mean():.2f}")
print(f"  Median: ${submission['price'].median():.2f}")

# Save submission
submission.to_csv('submission.csv', index=False)
print("\n" + "="*60)
print("Submission saved to 'submission.csv'")
print("="*60)
print("\nFirst 10 predictions:")
print(submission.head(10))
print("\nLast 10 predictions:")
print(submission.tail(10))

Loading data...
Train shape: (75000, 4)
Test shape: (75000, 3)

Preprocessing training data...
Starting preprocessing... Shape: (75000, 4)
Before filtering: 75000 rows
After filtering: 73914 rows
Preprocessing complete. Final shape: (73914, 120)

Preprocessing test data...
Starting preprocessing... Shape: (75000, 3)
Preprocessing complete. Final shape: (75000, 190)
✓ All 75000 test rows preserved

Feature count: 118
Training samples: 73914
Test samples: 75000

Training LightGBM & XGBoost Ensemble

--- Training LightGBM ---
Training until validation scores don't improve for 100 rounds
[100]	valid_0's mape: 1.62824
[200]	valid_0's mape: 1.4944
[300]	valid_0's mape: 1.43415
[400]	valid_0's mape: 1.4023
[500]	valid_0's mape: 1.38235
[600]	valid_0's mape: 1.37111
[700]	valid_0's mape: 1.35921
[800]	valid_0's mape: 1.35072
[900]	valid_0's mape: 1.34387
[1000]	valid_0's mape: 1.3372
[1100]	valid_0's mape: 1.33354
[1200]	valid_0's mape: 1.32864
[1300]	valid_0's mape: 1.32478
[1400]	valid_0's m

In [11]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.model_selection import KFold
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

def remove_emojis(text):
    if pd.isna(text):
        return ""
    text = str(text)
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002500-\U00002BEF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FAFF"
        "\U00002600-\U000026FF"
        "\U00002700-\U000027BF"
        "]+",
        flags=re.UNICODE
    )
    cleaned = emoji_pattern.sub('', text)
    cleaned = ' '.join(cleaned.split())
    return cleaned.strip()

def extract_fields_fixed(catalog_content):
    if pd.isna(catalog_content):
        return pd.Series({
            'item_name': "", 'value': None, 'unit': None, 
            'catalog_content_cleaned': "", 'product_desc': "", 
            'bullet_points': [], 'brand_info': "", 'bullet_point_count': 0
        })
    
    catalog_content = str(catalog_content)
    
    patterns = [
        r'Item\s*Name\s*:\s*(.+?)(?=Bullet|Product|Value|Description|$)',
        r'Item\s*Name\s*:\s*(.+?)$',
        r'Item\s*:\s*(.+?)(?=Bullet|Product|Value|Description|$)'
    ]
    
    item_name = ""
    for pattern in patterns:
        match = re.search(pattern, catalog_content, re.IGNORECASE | re.DOTALL)
        if match:
            item_name = match.group(1).strip()
            break
    
    value_patterns = [
        r'Value\s*:\s*([\d.,]+)',
        r'Size\s*:\s*([\d.,]+)',
        r'Weight\s*:\s*([\d.,]+)'
    ]
    
    value = None
    for pattern in value_patterns:
        match = re.search(pattern, catalog_content, re.IGNORECASE)
        if match:
            try:
                value = float(match.group(1).replace(',', '').strip())
                break
            except:
                continue
    
    unit_patterns = [
        r'Unit\s*:\s*(.+?)(?=\s*(?:Bullet|Product|Value|Description|$))',
        r'Size\s*:\s*[^:]*\s*([a-zA-Z]+)',
        r'Weight\s*:\s*[^:]*\s*([a-zA-Z]+)'
    ]
    
    unit = None
    for pattern in unit_patterns:
        match = re.search(pattern, catalog_content, re.IGNORECASE)
        if match:
            unit = match.group(1).strip()
            break
    
    desc_match = re.search(r'Product\s*Description\s*:\s*(.+?)(?=Bullet|Value|Unit|$)',
                          catalog_content, re.IGNORECASE | re.DOTALL)
    product_desc = desc_match.group(1).strip() if desc_match else ""
    
    bullet_points = re.findall(r'Bullet\s*Point\s*\d*\s*:\s*(.+?)(?=Bullet\s*Point|Product\s*Description|Value|Unit|$)',
                              catalog_content, re.IGNORECASE | re.DOTALL)
    bullet_points = [bp.strip() for bp in bullet_points if bp.strip()]
    bullet_point_count = len(bullet_points)
    
    brand_keywords = re.findall(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)*)\b', item_name)
    brand_info = ' '.join(brand_keywords[:3]) if brand_keywords else ""
    
    catalog_content_cleaned = re.sub(
        r'Item\s*Name\s*:.*?(?=Bullet|Product|Value|$)|Value\s*:.*?(?=Unit|Bullet|Product|$)|Unit\s*:.*?(?=Bullet|Product|$)|Product\s*Description\s*:.*?(?=Bullet|Value|Unit|$)',
        '', catalog_content, flags=re.IGNORECASE | re.DOTALL
    ).strip()
    
    return pd.Series({
        'item_name': item_name,
        'value': value,
        'unit': unit,
        'catalog_content_cleaned': catalog_content_cleaned,
        'product_desc': product_desc,
        'bullet_points': bullet_points,
        'brand_info': brand_info,
        'bullet_point_count': bullet_point_count
    })

def standardize_units(row):
    unit = row['unit']
    value = row['value']
    
    if pd.isna(unit) or pd.isna(value):
        return pd.Series({'standardized_value': value, 'standardized_unit': unit})
    
    try:
        value_float = float(value)
    except:
        return pd.Series({'standardized_value': value, 'standardized_unit': unit})
    
    unit_lower = str(unit).lower().strip()
    
    conversion_map = {
        'ounce': ('grams', 28.3495), 'ounces': ('grams', 28.3495), 'oz': ('grams', 28.3495),
        'gram': ('grams', 1), 'grams': ('grams', 1), 'g': ('grams', 1),
        'kilogram': ('grams', 1000), 'kilograms': ('grams', 1000), 'kg': ('grams', 1000), 'kgs': ('grams', 1000),
        'fl oz': ('ml', 29.5735), 'fluid ounce': ('ml', 29.5735), 'fluid ounces': ('ml', 29.5735),
        'floz': ('ml', 29.5735), 'fl. oz': ('ml', 29.5735),
        'ml': ('ml', 1), 'millilitre': ('ml', 1), 'milliliter': ('ml', 1),
        'liter': ('ml', 1000), 'liters': ('ml', 1000), 'litre': ('ml', 1000), 'litres': ('ml', 1000), 'l': ('ml', 1000),
        'pound': ('grams', 453.592), 'lb': ('grams', 453.592), 'lbs': ('grams', 453.592),
        'count': ('Count', 1), 'ct': ('Count', 1), 'each': ('Count', 1), 'piece': ('Count', 1), 'pack': ('Count', 1)
    }
    
    for unit_pattern, (target_unit, multiplier) in conversion_map.items():
        if unit_pattern in unit_lower:
            return pd.Series({
                'standardized_value': round(value_float * multiplier, 2), 
                'standardized_unit': target_unit
            })
    
    return pd.Series({'standardized_value': value_float, 'standardized_unit': unit})

def extract_advanced_price_features(item_name, desc, bullet_points):
    combined_text = ""
    if isinstance(item_name, str):
        combined_text += item_name.lower() + " "
    if isinstance(desc, str):
        combined_text += desc.lower()
    
    bullet_text = " ".join(bullet_points) if bullet_points else ""
    if bullet_text:
        combined_text += " " + bullet_text.lower()
    
    premium_terms = {
        'premium': 2, 'gourmet': 3, 'artisan': 3, 'luxury': 3, 'deluxe': 2,
        'supreme': 2, 'elite': 2, 'finest': 2, 'signature': 2, 'reserve': 2,
        'imported': 1, 'handmade': 2, 'handcrafted': 2, 'boutique': 2
    }
    
    economy_terms = {
        'value': -2, 'economy': -2, 'budget': -2, 'affordable': -1, 'cheap': -3,
        'save': -1, 'discount': -2, 'sale': -2, 'bargain': -2, 'low_price': -2
    }
    
    premium_score = sum(weight for term, weight in premium_terms.items() if re.search(r'\b' + term + r'\b', combined_text))
    economy_score = sum(weight for term, weight in economy_terms.items() if re.search(r'\b' + term + r'\b', combined_text))
    
    price_indicator = premium_score + economy_score
    
    return pd.Series({
        'premium_score': premium_score,
        'economy_score': economy_score,
        'price_indicator': price_indicator,
        'has_premium': int(premium_score >= 2),
        'has_economy': int(economy_score <= -2),
        'is_high_end': int(price_indicator >= 3),
        'is_budget': int(price_indicator <= -3)
    })

def extract_product_category_features(item_name, desc):
    combined = ""
    if isinstance(item_name, str):
        combined += item_name.lower() + " "
    if isinstance(desc, str):
        combined += desc.lower()
    
    categories = {
        'beverages': r'juice|soda|drink|beverage|water|sparkling|tea|coffee',
        'snacks': r'chips|crackers|nuts|popcorn|pretzel|snack|trail mix|granola bar',
        'canned_goods': r'canned|can|tin|jar|bottle|preserved',
        'frozen_foods': r'frozen|freeze|ice|frozen food',
        'bakery': r'bread|cookie|cake|pastry|bake|muffin|bagel',
        'dairy': r'milk|cheese|yogurt|cream|butter|dairy',
        'meat_poultry': r'beef|chicken|pork|meat|sausage|bacon|ham|turkey',
        'seafood': r'fish|salmon|tuna|shrimp|seafood|crab|lobster',
        'produce': r'fruit|vegetable|apple|banana|orange|lettuce|tomato|potato',
        'condiments': r'sauce|ketchup|mustard|mayo|dressing|spice|seasoning',
        'cereal_grains': r'cereal|granola|oatmeal|breakfast|rice|pasta|noodle'
    }
    
    features = {}
    for category, pattern in categories.items():
        count = len(re.findall(pattern, combined))
        features[f'is_{category}'] = int(count > 0)
        features[f'{category}_count'] = count
    
    return pd.Series(features)

def extract_text_quality_features(text):
    if not isinstance(text, str) or not text.strip():
        return pd.Series({
            'text_quality_score': 0,
            'word_count': 0,
            'unique_word_ratio': 0,
            'avg_word_length': 0
        })
    
    words = text.split()
    word_count = len(words)
    unique_words = len(set(words))
    avg_word_length = np.mean([len(word) for word in words]) if words else 0
    
    quality_terms = r'premium|quality|select|choice|prime|superior|excellent|finest'
    quality_count = len(re.findall(quality_terms, text.lower()))
    
    text_quality_score = (unique_words / max(word_count, 1)) * avg_word_length + quality_count
    
    return pd.Series({
        'text_quality_score': text_quality_score,
        'word_count': word_count,
        'unique_word_ratio': unique_words / max(word_count, 1),
        'avg_word_length': avg_word_length
    })

def extract_bullet_analysis(bullet_points):
    if not bullet_points:
        return pd.Series({
            'bullet_quality': 0,
            'bullet_specificity': 0,
            'avg_bullet_length': 0
        })
    
    bullet_text = " ".join(bullet_points).lower()
    
    quality_terms = r'easy|quick|convenient|fast|simple|benefit|advantage|feature'
    quality_score = len(re.findall(quality_terms, bullet_text))
    
    specificity_terms = r'\d+\.?\d*|percent|%|times|years|hours|minutes'
    specificity_score = len(re.findall(specificity_terms, bullet_text))
    
    avg_bullet_length = np.mean([len(bp) for bp in bullet_points])
    
    return pd.Series({
        'bullet_quality': quality_score,
        'bullet_specificity': specificity_score,
        'avg_bullet_length': avg_bullet_length
    })

def calculate_smape(actual, predicted):
    actual = np.array(actual)
    predicted = np.array(predicted)
    return 100 * np.mean(2 * np.abs(predicted - actual) / (np.abs(actual) + np.abs(predicted)))

def fixed_preprocessing(df, is_train=True, scaler=None, power_scaler=None, 
                       tfidf_name=None, tfidf_desc=None, svd_name=None, svd_desc=None, 
                       train_median_value=None, train_unit_cols=None):
    
    print(f"Starting preprocessing... Shape: {df.shape}")
    initial_rows = len(df)
    
    df['catalog_content'] = df['catalog_content'].apply(remove_emojis)
    
    extracted_fields = df['catalog_content'].apply(extract_fields_fixed)
    df = pd.concat([df, extracted_fields], axis=1)
    
    df[['standardized_value', 'standardized_unit']] = df.apply(standardize_units, axis=1)
    
    if is_train:
        train_median_value = df['standardized_value'].median()
    
    df['standardized_unit'] = df['standardized_unit'].fillna('grams')
    df['standardized_value'] = df['standardized_value'].fillna(train_median_value)
    df['item_name'] = df['item_name'].fillna('')
    
    # One-hot encode units
    df = pd.get_dummies(df, columns=['standardized_unit'], prefix='unit', dtype=int)
    
    # CRITICAL FIX: Align unit columns between train and test
    if is_train:
        train_unit_cols = [col for col in df.columns if col.startswith('unit_')]
    else:
        # For test set, ensure we have the same unit columns as training
        current_unit_cols = [col for col in df.columns if col.startswith('unit_')]
        
        # Add missing unit columns (set to 0)
        for col in train_unit_cols:
            if col not in df.columns:
                df[col] = 0
        
        # Remove extra unit columns not in training
        for col in current_unit_cols:
            if col not in train_unit_cols:
                df = df.drop(columns=[col])
    
    df = df.drop(['value', 'unit', 'catalog_content'], axis=1, errors='ignore')
    
    df['has_catalog_content'] = ((~df['catalog_content_cleaned'].isna()) & 
                                  (df['catalog_content_cleaned'].str.strip() != '')).astype(int)
    df['has_product_desc'] = (df['product_desc'].str.len() > 0).astype(int)
    df['has_brand_info'] = (df['brand_info'].str.len() > 0).astype(int)
    df['has_bullet_points'] = (df['bullet_point_count'] > 0).astype(int)
    
    if is_train:
        scaler = StandardScaler()
        df['value_scaled'] = scaler.fit_transform(df[['standardized_value']])
        
        power_scaler = PowerTransformer(method='yeo-johnson')
        df['value_power'] = power_scaler.fit_transform(df[['standardized_value']])
        
        df['value_log'] = np.log1p(df['standardized_value'])
        df['value_sqrt'] = np.sqrt(df['standardized_value'])
    else:
        df['value_scaled'] = scaler.transform(df[['standardized_value']])
        df['value_power'] = power_scaler.transform(df[['standardized_value']])
        df['value_log'] = np.log1p(df['standardized_value'])
        df['value_sqrt'] = np.sqrt(df['standardized_value'])
    
    df = df.drop(columns=['standardized_value'], errors='ignore')
    
    df['item_name_clean'] = df['item_name'].fillna('').apply(lambda x: re.sub(r'[^\w\s]', ' ', str(x)).strip())
    df['desc_clean'] = df['catalog_content_cleaned'].fillna('').apply(lambda x: re.sub(r'[^\w\s]', ' ', str(x)).strip())
    df['product_desc_clean'] = df['product_desc'].fillna('').apply(lambda x: re.sub(r'[^\w\s]', ' ', str(x)).strip())
    
    print("Extracting advanced features...")
    
    price_features_df = df.apply(lambda row: extract_advanced_price_features(
        row['item_name'], row['catalog_content_cleaned'], row['bullet_points']), axis=1)
    df = pd.concat([df, price_features_df], axis=1)
    
    category_df = df.apply(lambda row: extract_product_category_features(
        row['item_name'], row['catalog_content_cleaned']), axis=1)
    df = pd.concat([df, category_df], axis=1)
    
    text_quality_item = df['item_name'].apply(extract_text_quality_features)
    text_quality_item.columns = [f'item_{col}' for col in text_quality_item.columns]
    df = pd.concat([df, text_quality_item], axis=1)
    
    text_quality_desc = df['catalog_content_cleaned'].apply(extract_text_quality_features)
    text_quality_desc.columns = [f'desc_{col}' for col in text_quality_desc.columns]
    df = pd.concat([df, text_quality_desc], axis=1)
    
    bullet_analysis_df = df['bullet_points'].apply(extract_bullet_analysis)
    df = pd.concat([df, bullet_analysis_df], axis=1)
    
    df['item_name_length'] = df['item_name'].str.len().fillna(0)
    df['desc_length'] = df['catalog_content_cleaned'].str.len().fillna(0)
    df['product_desc_length'] = df['product_desc'].str.len().fillna(0)
    df['total_text_length'] = df['item_name_length'] + df['desc_length'] + df['product_desc_length']
    
    df['text_richness'] = df['total_text_length'] / (df['bullet_point_count'] + 1)
    
    df['combined_text'] = (df['item_name_clean'] + " " + 
                          df['desc_clean'] + " " + 
                          df['product_desc_clean'])
    
    if is_train:
        tfidf_name = TfidfVectorizer(
            max_features=300, 
            min_df=2, 
            max_df=0.95,
            ngram_range=(1, 2),
            sublinear_tf=True,
            stop_words='english'
        )
        tfidf_name_matrix = tfidf_name.fit_transform(df['item_name_clean'].fillna(''))
        
        svd_name = TruncatedSVD(n_components=50, random_state=42)
        name_embeddings = svd_name.fit_transform(tfidf_name_matrix)
        
        tfidf_desc = TfidfVectorizer(
            max_features=400,
            min_df=2,
            max_df=0.95,
            ngram_range=(1, 2),
            sublinear_tf=True,
            stop_words='english'
        )
        tfidf_desc_matrix = tfidf_desc.fit_transform(df['combined_text'].fillna(''))
        
        svd_desc = TruncatedSVD(n_components=80, random_state=42)
        desc_embeddings = svd_desc.fit_transform(tfidf_desc_matrix)
    else:
        tfidf_name_matrix = tfidf_name.transform(df['item_name_clean'].fillna(''))
        name_embeddings = svd_name.transform(tfidf_name_matrix)
        
        tfidf_desc_matrix = tfidf_desc.transform(df['combined_text'].fillna(''))
        desc_embeddings = svd_desc.transform(tfidf_desc_matrix)
    
    name_cols = [f'name_emb_{i}' for i in range(50)]
    name_df = pd.DataFrame(name_embeddings, columns=name_cols, index=df.index)
    
    desc_cols = [f'desc_emb_{i}' for i in range(80)]
    desc_df = pd.DataFrame(desc_embeddings, columns=desc_cols, index=df.index)
    
    df = pd.concat([df, name_df, desc_df], axis=1)
    
    text_cols_to_drop = ['item_name', 'catalog_content_cleaned', 'product_desc', 
                        'bullet_points', 'brand_info', 'item_name_clean', 
                        'desc_clean', 'product_desc_clean', 'combined_text', 'image_link']
    df = df.drop(columns=[col for col in text_cols_to_drop if col in df.columns], errors='ignore')
    
    final_rows = len(df)
    print(f"Preprocessing complete. Final shape: {df.shape}")
    print(f"Rows: Initial={initial_rows}, Final={final_rows}, Dropped={initial_rows - final_rows}")
    
    if is_train:
        return df, scaler, power_scaler, tfidf_name, tfidf_desc, svd_name, svd_desc, train_median_value, train_unit_cols
    else:
        return df

# ==================== MAIN EXECUTION ====================

print("Loading data...")
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

print(f"\nTarget price statistics:")
print(f"Min: ${train_df['price'].min():.2f}")
print(f"Max: ${train_df['price'].max():.2f}")
print(f"Mean: ${train_df['price'].mean():.2f}")
print(f"Median: ${train_df['price'].median():.2f}")

y_train_original = train_df['price'].copy()
y_train_transformed = np.log1p(train_df['price'])

print("\n" + "="*60)
print("PREPROCESSING DATA")
print("="*60)

train_processed, scaler, power_scaler, tfidf_name, tfidf_desc, svd_name, svd_desc, train_median_value, train_unit_cols = fixed_preprocessing(
    train_df.copy(), is_train=True
)

test_processed = fixed_preprocessing(
    test_df.copy(), is_train=False, scaler=scaler, power_scaler=power_scaler,
    tfidf_name=tfidf_name, tfidf_desc=tfidf_desc,
    svd_name=svd_name, svd_desc=svd_desc,
    train_median_value=train_median_value, train_unit_cols=train_unit_cols
)

feature_cols = [col for col in train_processed.columns 
                if col not in ['sample_id', 'price']]

# Clean feature names to remove special characters that XGBoost doesn't allow
def clean_feature_names(df):
    new_cols = {}
    for col in df.columns:
        # Replace [, ], < with underscores
        new_col = col.replace('[', '_').replace(']', '_').replace('<', '_')
        new_col = new_col.replace('>', '_').replace(',', '_').replace(' ', '_')
        new_cols[col] = new_col
    return df.rename(columns=new_cols)

X_train = train_processed[feature_cols].fillna(0)
X_test = test_processed[feature_cols].fillna(0)

# Clean column names
X_train = clean_feature_names(X_train)
X_test = clean_feature_names(X_test)

print(f"\n{'='*60}")
print(f"TRAINING XGBOOST MODEL")
print(f"{'='*60}")
print(f"Features: {len(feature_cols)}")
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

n_folds = 5
kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)

xgb_test_preds = np.zeros(len(X_test))
xgb_train_preds = np.zeros(len(X_train))
fold_scores = []

xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mape',
    'learning_rate': 0.01,
    'max_depth': 10,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'gamma': 0.1,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1
}

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    print(f"\n--- Fold {fold+1}/{n_folds} ---")
    
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train_transformed.iloc[train_idx], y_train_transformed.iloc[val_idx]
    
    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=2000,
        evals=[(dval, 'eval')],
        early_stopping_rounds=100,
        verbose_eval=False
    )
    
    val_pred = model.predict(dval)
    xgb_train_preds[val_idx] = val_pred
    
    smape = calculate_smape(np.expm1(y_val.values), np.expm1(val_pred))
    fold_scores.append(smape)
    print(f"Fold {fold+1} SMAPE: {smape:.4f} (iterations: {model.best_iteration})")
    
    dtest = xgb.DMatrix(X_test)
    fold_test_pred = model.predict(dtest)
    xgb_test_preds += fold_test_pred / n_folds

final_train_pred = np.expm1(xgb_train_preds)
final_test_pred = np.expm1(xgb_test_preds)

final_smape = calculate_smape(y_train_original, final_train_pred)

print(f"\n{'='*60}")
print(f"XGBOOST CV SMAPE: {np.mean(fold_scores):.4f} (+/- {np.std(fold_scores):.4f})")
print(f"FINAL TRAINING SMAPE: {final_smape:.4f}")
print(f"{'='*60}")

submission = pd.DataFrame({
    'sample_id': test_processed['sample_id'],
    'price': final_test_pred
})

submission['price'] = submission['price'].clip(lower=0.01)

print(f"\n{'='*60}")
print(f"SUBMISSION DETAILS")
print(f"{'='*60}")
print(f"Total predictions: {len(submission)}")
print(f"Expected test samples: {len(test_df)}")
print(f"Match: {len(submission) == len(test_df)}")
print(f"\nPrice statistics:")
print(f"  Min: ${submission['price'].min():.2f}")
print(f"  Max: ${submission['price'].max():.2f}")
print(f"  Mean: ${submission['price'].mean():.2f}")
print(f"  Median: ${submission['price'].median():.2f}")

submission.to_csv('submission.csv', index=False)
print(f"\n✅ Submission saved to 'submission.csv'")

print(f"\nFirst 10 predictions:")
print(submission.head(10))
print(f"\nLast 10 predictions:")
print(submission.tail(10))

Loading data...
Train shape: (75000, 4)
Test shape: (75000, 3)

Target price statistics:
Min: $0.13
Max: $2796.00
Mean: $23.65
Median: $14.00

PREPROCESSING DATA
Starting preprocessing... Shape: (75000, 4)
Extracting advanced features...
Preprocessing complete. Final shape: (75000, 217)
Rows: Initial=75000, Final=75000, Dropped=0
Starting preprocessing... Shape: (75000, 3)
Extracting advanced features...
Preprocessing complete. Final shape: (75000, 216)
Rows: Initial=75000, Final=75000, Dropped=0

TRAINING XGBOOST MODEL
Features: 215
Training samples: 75000
Test samples: 75000

--- Fold 1/5 ---
Fold 1 SMAPE: 53.2470 (iterations: 1999)

--- Fold 2/5 ---
Fold 2 SMAPE: 52.1597 (iterations: 1999)

--- Fold 3/5 ---
Fold 3 SMAPE: 52.6284 (iterations: 1999)

--- Fold 4/5 ---
Fold 4 SMAPE: 51.5296 (iterations: 1997)

--- Fold 5/5 ---
Fold 5 SMAPE: 52.3762 (iterations: 1998)

XGBOOST CV SMAPE: 52.3882 (+/- 0.5630)
FINAL TRAINING SMAPE: 52.3882

SUBMISSION DETAILS
Total predictions: 75000
Expect